# Boundary proposals: validation dari cache

Buka notebook dari root **SongMap**, kernel Python mana pun. Proses evaluasi memakai interpreter `.venv-songformer311` milik Section-Annotations secara eksplisit. Hanya head Core ML pada CPU yang dijalankan; encoder, decoding audio, training, dan anotasi ulang tidak dilakukan. Memakai validation 32 lagu yang sudah dibekukan, model palette6, dan cache fitur sebelumnya.

Proposal tetap simulasi. Manual label adalah referensi section layak; banyak timestamp referensinya mungkin masih berasal dari Apple. Jadi laporan ini mengukur dampak terhadap label, bukan membuktikan akurasi boundary independen. Tidak mengubah aplikasi atau dataset.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, subprocess, sys
assert Path('scripts/evaluate_boundary_proposals.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=Path('../Section-Annotations').resolve()
assert (SONGSTRUCT_ROOT/'finetuning/config.json').is_file()
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Environment tidak ditemukan: {RUN_PYTHON}'
preflight=[str(RUN_PYTHON),'-c',"import importlib.util,sys; missing=[p for p in ('numpy','torch','coremltools') if importlib.util.find_spec(p) is None]; print('Evaluation Python:',sys.executable); sys.exit('Dependency belum tersedia: '+', '.join(missing)) if missing else print('Dependency check OK')"]
subprocess.run(preflight,check=True)
output=Path('outputs.local')/('boundary-validation-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
print('Notebook kernel:',sys.executable)
print('Evaluation Python:',RUN_PYTHON)
print('SongStruct:',SONGSTRUCT_ROOT)
print('Output:',output)


In [ ]:
def run_validation(limit,name):
    output.mkdir(parents=True,exist_ok=True)
    log=output/(name+'.log')
    command=[str(RUN_PYTHON),'-u','scripts/evaluate_boundary_proposals.py',
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--output-dir',str(output),'--limit',str(limit)]
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Gagal ({status}); lihat {log.resolve()}')
run_validation(1,'one')


In [ ]:
run_validation(0,'all')
r=json.loads((output/'validation-all.json').read_text())
for key in ('status','tracks','full_validation','baseline_prediction_mismatches',
            'tracks_with_proposals','total_proposals','simulated_total_new_splits',
            'baseline','candidate','boundary_accuracy_validated','limitations'):
    print(key+':',json.dumps(r[key],indent=2))
print('Bagikan:',(output/'validation-all.json').resolve())


`baseline_prediction_mismatches` harus kosong: prediksi head dari cache harus mereproduksi laporan palette6 sebelumnya sebelum hasil usulan ditafsirkan.

Bandingkan baseline dan candidate hanya di dalam laporan ini, karena metrik vote berdasar durasi berbeda dari pooling probabilitas pada evaluasi terdahulu. `eligible_duration_label_agreement` mengukur tumpang tindih label dengan target layak; catatan pending/held tidak menjadi ground truth. Lihat perubahan per lagu dan daftar usulan. `boundary_accuracy_validated` tetap false tanpa referensi boundary independen. Threshold belum boleh dianggap siap produksi hanya karena satu skor naik.